# **Induction heads: practice**

An induction head is an attention head that, on a repeat "…[A][B] … [A]", attends to the token [B] that followed the earlier [A] and copies it into the prediction. [Olsson et al., 2022](https://arxiv.org/abs/2209.11895) define such heads by their behavior on a repeated random token sequence; [Elhage et al., 2021](https://transformer-circuits.pub/2021/framework/index.html) took them apart in a two-layer attention-only model: a previous-token head in the first layer and an induction head in the second, connected by K-composition.

In this practice you take such a model — `attn-only-2l` from TransformerLens (2 layers of 8 attention heads, no MLP) — and find both heads in it in three ways: by the attention pattern, by ablation, and by the weights.

The tasks are numbered; you enter the answers in the lesson "Induction heads: Practice".

## **Set up**
A CPU is enough: the model is small.

In [ ]:
!pip install transformer-lens==3.9.0 -q

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)  # forward passes only
model = HookedTransformer.from_pretrained("attn-only-2l", device="cpu")
print(model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_model)

## **Data: a repeated random sequence**

Take 20 sequences of 50 random tokens, repeat each one twice and put a BOS token in front. In the first half the next token cannot be guessed: the tokens are random. In the second half every token has already appeared, and a model with induction heads can predict it by looking at what followed the earlier occurrence of the current token.

The loss is computed on predicting tokens 2–50 of each half: the first token of the second half still follows from nothing.

In [ ]:
L, B = 50, 20
gen = torch.Generator().manual_seed(0)
rand = torch.randint(1000, 10000, (B, L), generator=gen)
bos = torch.full((B, 1), model.tokenizer.bos_token_id)
tokens = torch.cat([bos, rand, rand], dim=1)  # [20, 1 + 2*50]


def half_losses(logits):
    """Mean loss of predicting tokens 2–50 of the first and the second half."""
    lp = utils.lm_cross_entropy_loss(logits, tokens, per_token=True)  # lp[:, j] is the loss of predicting token j + 1
    return ...  # Your code here


logits, cache = model.run_with_cache(tokens)
first, second = half_losses(logits)
print(f"loss: first half {first:.2f}, second half {second:.2f}")

In [ ]:
lp = utils.lm_cross_entropy_loss(logits, tokens, per_token=True).mean(0)
plt.figure(figsize=(8, 3))
plt.plot(range(1, 2 * L + 1), lp)  # position 0 is BOS, the first half is positions 1…50
plt.axvline(L + 0.5, color="gray", linestyle="--")
plt.xlabel("position of the predicted token")
plt.ylabel("loss, mean over 20 sequences")
plt.title("Loss on a repeated random sequence")
plt.show()

**Task 1. What is the mean loss on the first half? Round to two decimals.**

**Task 2. What is the mean loss on the second half? Round to two decimals.**

## **Attention pattern: two scores per head**

For each head compute two quantities on the second half, averaged over positions and sequences:

- **previous-token score** — the attention weight from position $i$ to position $i-1$;
- **prefix-matching score** — the attention weight from position $i$ to position $i-L+1$: where the token that followed the earlier occurrence of the current token stands.

A previous-token head has a high first score, an induction head a high second one.

In [ ]:
prev_score = torch.zeros(2, model.cfg.n_heads)
prefix_score = torch.zeros(2, model.cfg.n_heads)
for layer in range(2):
    pattern = cache["pattern", layer]  # [batch, head, query position, key position]
    # the last L diagonal entries are the second-half queries, positions L+1…2L
    prev_score[layer] = ...  # Your code here
    prefix_score[layer] = ...  # Your code here

for layer in range(2):
    print(f"layer {layer}: prev", [round(x, 2) for x in prev_score[layer].tolist()])
    print(f"layer {layer}: prefix", [round(x, 2) for x in prefix_score[layer].tolist()])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (layer, head) in zip(axes, [(0, 3), (1, 6)]):
    ax.imshow(cache["pattern", layer][0, head], cmap="Blues")
    ax.set_title(f"head {layer}.{head}, sequence 0")
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
plt.show()

**Task 3. Which head has the largest prefix-matching score?** Write a head as "layer.head", for example 1.2.

**Task 4. What is that prefix-matching score? Round to two decimals.**

**Task 5. Which layer-0 head has the largest previous-token score?**

**Task 6. What is that previous-token score? Round to two decimals.**

## **Head ablation**

The attention pattern shows where a head looks, not whether the prediction needs it. Check by intervention: zero the output of one head (`hook_z`) and see how the loss on the second half changes. This is zero ablation; the lesson on zero, mean and resample ablations explains how it differs from mean and resample ablation.

In [ ]:
def ablated_losses(layer, head):
    """First- and second-half loss with the output of one head zeroed."""
    def zero_head(z, hook):
        ...  # Your code here
        return z
    out = model.run_with_hooks(tokens, fwd_hooks=[(utils.get_act_name("z", layer), zero_head)])
    return half_losses(out)


abl = torch.zeros(2, model.cfg.n_heads)
for layer in range(2):
    for head in range(model.cfg.n_heads):
        abl[layer, head] = ablated_losses(layer, head)[1]
for layer in range(2):
    print(f"layer {layer}:", [round(x, 2) for x in abl[layer].tolist()])

**Task 7. What is the loss on the second half after ablating the head from task 3? Round to two decimals.**

**Task 8. Ablating which layer-0 head raises the second-half loss the most?**

## **Composition from the weights**

Elhage et al. measure how much a second-layer head reads the output of a first-layer head by the Frobenius norm of a matrix product: for K-composition $\|W_{QK}^{h_2}W_{OV}^{h_1}\|_F / (\|W_{QK}^{h_2}\|_F\,\|W_{OV}^{h_1}\|_F)$, for Q- and V-composition the same with $W_{QK}^{h_2\,T}$ and $W_{OV}^{h_2}$. In TransformerLens this is `model.all_composition_scores(mode)`. The score needs no data: it is computed from the weights. In the paper, Elhage et al. subtract from this score its mean for random matrices of the same shape; `all_composition_scores` returns the score without this correction, so it is compared with other scores of the same model.

In [ ]:
h1, h2 = (0, 3), (1, 6)  # previous-token head and induction head
for mode in "QKV":
    scores = model.all_composition_scores(mode)  # [layer1, head1, layer2, head2]
    ...  # Your code here

**Task 9. Which type of composition between heads 0.3 and 1.6 gives the largest score?**

## **Conclusion**

- On a repeated random sequence the second-half loss is several times lower than the first-half loss: the model copies the continuation from the context.
- The attention pattern singles out two heads: a previous-token head in layer 0 and an induction head in layer 1 with a prefix-matching score of about 0.7.
- Ablation confirms that both are needed for copying: without either of them the second-half loss rises several times, and the layer-0 head with the largest previous-token score is the same one whose ablation raises the loss the most.
- The weights show the channel that connects the two heads: their K-composition score is higher than the Q- and V-composition scores, as the circuit of Elhage et al. predicts.